# 05. Advanced Model Experiments: Structured Roadmap

Fundamentals of Natural Language / NLP-I, Universitat Autonoma de Barcelona, academic year 2025-2026. Team 10: Phoebe Iglesias, David Redrejo, and Pau Rossell.

At this point we stop ourselves from adding advanced models randomly. We already have EDA, preprocessing decisions, classical baselines, retrieval, CLS pooling, and mean pooling. This notebook turns those results into a controlled experimental roadmap.

## 1. What We Learned So Far

**From EDA.** The dataset is short-literal ICD category prediction, not long-document ICD coding. It has 36 categories, strong class imbalance, duplicate literals, and some same-literal/different-code ambiguities. Leaderboard literals have similar length distributions, but possible distribution shift remains because we only observe text length and surface patterns.

**From preprocessing.** Biomedical Spanish text is fragile. We preserve case, accents, punctuation, digits, and abbreviations for RoBERTa. Light whitespace cleanup is the final pipeline; stronger normalization is only an ablation for classical baselines.

**From classical baselines.** Majority baseline gives 0.125 accuracy and almost zero macro F1. Character TF-IDF and word TF-IDF are strong: they reach around 0.52 accuracy and show that surface lexical patterns matter. Retrieval is intuitive but weaker than classifiers because identical or similar literals can still map to different categories.

**From CLS pooling.** RoBERTa CLS achieved the best validation accuracy so far: 0.5693. This suggests that the Spanish biomedical-clinical pretrained language model adds value beyond sparse TF-IDF features.

**From mean pooling.** Mean pooling did not beat CLS in accuracy, but it slightly improved macro F1. This suggests that pooling choices affect class balance and not only the overall score.

In [ ]:
import pandas as pd

model_summary = pd.DataFrame([
    {'model': 'v00_majority_baseline', 'accuracy': 0.125182, 'macro_f1': 0.006181, 'weighted_f1': 0.027854},
    {'model': 'v01_tfidf_char_logreg', 'accuracy': 0.522628, 'macro_f1': 0.402554, 'weighted_f1': 0.494943},
    {'model': 'v02_tfidf_word_svm', 'accuracy': 0.520073, 'macro_f1': 0.474196, 'weighted_f1': 0.514018},
    {'model': 'v03_similarity_retrieval_baseline', 'accuracy': 0.497445, 'macro_f1': 0.462789, 'weighted_f1': 0.496120},
    {'model': 'v04_roberta_cls', 'accuracy': 0.569343, 'macro_f1': 0.494329, 'weighted_f1': 0.554347},
    {'model': 'v05_roberta_mean', 'accuracy': 0.564599, 'macro_f1': 0.496567, 'weighted_f1': 0.549541},
])
model_summary


## 2. Weaknesses That Remain

- **Class imbalance:** some categories remain rare, and rare labels such as `A`, `W`, and `X` have zero recall in both CLS and mean pooling validation reports.
- **Ambiguous short literals:** nearest-neighbor analysis showed cases where exact or near-exact literals point to different categories. Short text lacks the clinical context that would resolve ambiguity.
- **Categories with low recall:** RoBERTa improves accuracy but does not solve minority categories. Macro F1 remains close to classical baselines.
- **Similar categories confused:** broad ICD categories can share surface terminology, especially when procedures, diagnoses, and history codes are written compactly.
- **Possible leaderboard distribution shift:** train and leaderboard look similar in length, but we do not observe leaderboard labels, so lexical or category distribution shift is still possible.
- **Overfitting/underfitting:** training loss keeps decreasing after the best validation epoch while validation loss rises, so later experiments should manage regularization and schedules.

In [ ]:
cls_per_class = pd.read_csv('../outputs/metrics/v04_roberta_cls_per_class_metrics.csv')
mean_per_class = pd.read_csv('../outputs/metrics/v05_roberta_mean_per_class_metrics.csv')
cls_per_class.sort_values('recall').head(10)


## 3. Candidate Improvements

The table below is the roadmap. Each candidate is tied to evidence from EDA, the course material, the ICD coding survey, or our observed model behavior. The goal is not to implement everything; the goal is to choose experiments that answer a specific question.

In [ ]:
roadmap = pd.read_csv('../reports/tables/advanced_experiment_roadmap.csv')
roadmap


## 4. Decisions

**Implement next.** We prioritize class-weighted loss, learning-rate tuning, warmup scheduler, dropout tuning, ensembling, and calibration/confidence analysis. These are connected to observed weaknesses and have low-to-medium implementation cost.

**Maybe.** Focal loss, max-length tuning, freezing/unfreezing, label smoothing, and safe data augmentation are plausible but should be tested only after the first priority experiments.

**Future work.** Layer-wise learning-rate decay and pseudo-labeling are interesting but more complex or riskier. They are better framed as future work unless we have time and a stable validation protocol.

## 5. Why This Roadmap Fits the Course Story

This roadmap follows the project story from the course: corpora and annotation analysis first, then text processing and sparse vector-space baselines, then pretrained Transformer models, and finally controlled improvement experiments.

The survey helped us understand why ICD coding is hard: imbalance, terminology, hierarchy, and interpretability. Our Kaggle task is simpler than full multi-label ICD coding, but the same ideas still guide our decisions. The next experiments should therefore be justified by the data and by the course concepts, not by trial-and-error model stacking.